In [0]:
%pip install --upgrade pydantic
%pip uninstall -y evidently
%pip install evidently
%pip install --upgrade evidently

In [0]:
%restart_python

In [0]:
# kDescarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import mlflow
import mlflow.sklearn

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

for params in param_grid:
    with mlflow.start_run():
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        #mlflow.sklearn.log_model(clf, "model")

In [0]:
mlflow.sklearn.log_model(clf, name="model")

In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pydantic

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

best_auc = 0
best_run_id = None

for params in param_grid:
    with mlflow.start_run() as run:
        clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        y_proba = clf.predict_proba(X_test)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_proba)
        cm = confusion_matrix(y_test, y_pred)
        report = classification_report(y_test, y_pred, output_dict=True)
        signature = infer_signature(X_test, y_pred)
        
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
        
        if auc > best_auc:
            best_auc = auc
            best_run_id = run.info.run_id

In [0]:
# Descarga dataset de ejemplo
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature
import pydantic

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

# Define hiperparámetros para experimentar
param_grid = [
    {"n_estimators": 50, "max_depth": 3},
    {"n_estimators": 100, "max_depth": 5},
    {"n_estimators": 150, "max_depth": 7},
    {"n_estimators": 200, "max_depth": 9}
]

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

best_auc = 0
best_run_id = None

with mlflow.start_run(run_name="parent_run") as parent_run:
    mlflow.log_param("experiment_type", "RandomForest_Hyperparameter_Search")
    for params in param_grid:
        with mlflow.start_run(run_name=f"child_run_{params['n_estimators']}_{params['max_depth']}", nested=True):
            clf = RandomForestClassifier(n_estimators=params["n_estimators"], max_depth=params["max_depth"], random_state=42)
            clf.fit(X_train, y_train)
            y_pred = clf.predict(X_test)
            y_proba = clf.predict_proba(X_test)[:, 1]
            acc = accuracy_score(y_test, y_pred)
            auc = roc_auc_score(y_test, y_proba)
            cm = confusion_matrix(y_test, y_pred)
            report = classification_report(y_test, y_pred, output_dict=True)
            signature = infer_signature(X_test, y_pred)
            
            mlflow.log_params(params)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("roc_auc", auc)
            mlflow.log_metric("precision", report["1"]["precision"])
            mlflow.log_metric("recall", report["1"]["recall"])
            mlflow.log_metric("f1_score", report["1"]["f1-score"])
            mlflow.sklearn.log_model(clf, signature=signature)
            mlflow.log_dict(report, "classification_report.json")
            mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")
            
            if auc > best_auc:
                best_auc = auc
                best_run_id = mlflow.active_run().info.run_id

In [0]:
%pip install optuna
import optuna
from sklearn.datasets import load_breast_cancer
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, classification_report
import mlflow
import mlflow.sklearn
from mlflow.models.signature import infer_signature

# Carga y prepara datos
data = load_breast_cancer()
df = pd.DataFrame(data.data, columns=data.feature_names)
df['target'] = data.target

X = df.drop('target', axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, stratify=y)

mlflow.set_experiment("/Users/mblancovillar@itba.edu.ar/Clase Prueba")

def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 200, step=50)
    max_depth = trial.suggest_int("max_depth", 3, 9, step=2)
    clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth, random_state=42)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    y_proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, y_proba)
    trial.set_user_attr("clf", clf)
    trial.set_user_attr("y_pred", y_pred)
    trial.set_user_attr("y_proba", y_proba)
    return auc

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=20)

# Log only the most promising trials (e.g., top 3)
top_trials = sorted(study.trials, key=lambda t: t.value, reverse=True)[:3]

for trial in top_trials:
    clf = trial.user_attrs["clf"]
    y_pred = trial.user_attrs["y_pred"]
    y_proba = trial.user_attrs["y_proba"]
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, output_dict=True)
    signature = infer_signature(X_test, y_pred)
    params = trial.params
    with mlflow.start_run(run_name=f"optuna_trial_{trial.number}"):
        mlflow.log_params(params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("roc_auc", auc)
        mlflow.log_metric("precision", report["1"]["precision"])
        mlflow.log_metric("recall", report["1"]["recall"])
        mlflow.log_metric("f1_score", report["1"]["f1-score"])
        mlflow.sklearn.log_model(clf, signature=signature)
        mlflow.log_dict(report, "classification_report.json")
        mlflow.log_dict({"confusion_matrix": cm.tolist()}, "confusion_matrix.json")

In [0]:
import numpy as np
# Restaurar el alias eliminado para compatibilidad
np.float_ = np.float64 

from evidently import Report
from evidently.presets import DataDriftPreset, ClassificationPreset


# Evidently analysis for the last trained model
y_pred = clf.predict(X_test)
y_proba = clf.predict_proba(X_test)[:, 1]

# Ensure prediction and proba have correct lengths
data_current = pd.DataFrame(X_test).assign(target=y_test)
if len(y_pred) == len(y_test):
    data_current["prediction"] = y_pred
if len(y_proba) == len(y_test):
    data_current["proba"] = y_proba

report = Report(metrics=[
    DataDriftPreset(),
    #ClassificationPreset()
])



In [0]:
data_current

In [0]:
y_train_pred = clf.predict(X_train)
data_reference = pd.DataFrame(X_train).assign(target=y_train, prediction=y_train_pred)


In [0]:
y_train_proba = clf.predict_proba(X_train)[:, 1]
data_reference["proba"] = y_train_proba


In [0]:
report.run(
    current_data=data_current,
    reference_data=data_reference
)

In [0]:
#
%pip install shap
import shap

# SHAP analysis for the last trained model
explainer = shap.Explainer(clf, X_train)
shap_values = explainer(X_test)

# SHAP summary plot
shap.summary_plot(shap_values.values, X_test, feature_names=X_test.columns)